# PatchCore WRN50 Memory-Bank And Feature Sweep

This notebook is the next improvement pass for the labeled `120k / 10k / 20k` WideResNet50-2 PatchCore workflow.

It is built around four concrete moves:

- larger memory-bank source coverage
- normal-only memory sampling
- `96 / 128` image size
- `layer1` inclusion sweeps

Why this notebook exists:

- threshold tuning improved the operating point, but it did not improve AUROC or AUPRC
- the current `50k` memory bank only needs about `64` source wafers for `layer2 + layer3`, and even fewer when `layer1` is included
- some of that source-image budget can be wasted if anomalous train rows are sampled before the memory bank is filtered to normals
- subtle local defects are still the likely failure mode, so more spatial detail and shallower features are the next reasonable bets

Default behavior is safe: this notebook shows the planned experiments first and only runs them if `RUN_EXPERIMENTS = True`.


In [ ]:
from __future__ import annotations

import json
import math
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import torch
from IPython.display import Markdown, display
from torch.utils.data import DataLoader


def resolve_notebook_root(start: Path | None = None) -> Path:
    start_path = (start or Path.cwd()).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'helpers' / 'patchcore_wrn50_modal.py').exists():
            return candidate
        nested = candidate / 'notebooks' / 'patchcore_wrn50_120k_labeled'
        if (nested / 'helpers' / 'patchcore_wrn50_modal.py').exists():
            return nested
    raise FileNotFoundError('Could not locate the WRN50 labeled notebook root.')


def resolve_project_root(start: Path | None = None) -> Path:
    start_path = (start or Path.cwd()).resolve()
    for candidate in [start_path, *start_path.parents]:
        if (candidate / 'notebooks').exists() and (candidate / 'data').exists():
            return candidate
    if os.environ.get('WM811K_DATA_ROOT') or os.environ.get('WM811K_OUTPUT_ROOT'):
        return start_path
    raise FileNotFoundError('Could not locate the project root.')


NOTEBOOK_ROOT = resolve_notebook_root()
PROJECT_ROOT = resolve_project_root(NOTEBOOK_ROOT)
HELPERS_ROOT = NOTEBOOK_ROOT / 'helpers'
if str(HELPERS_ROOT) not in sys.path:
    sys.path.insert(0, str(HELPERS_ROOT))

from patchcore_wrn50_modal import (
    DEFAULT_SPLIT_CONFIG,
    WaferArrayDataset,
    auto_find_raw_pickle,
    defect_type_summary,
    prepare_dataset,
    resolve_device,
    run_patchcore_variant,
    set_seed,
    split_summary_wide,
)


In [ ]:
DEFAULT_LOCAL_RAW_PICKLE = PROJECT_ROOT / 'data' / 'raw' / 'LSWMD.pkl'
RAW_PICKLE = os.environ.get('WM811K_RAW_PICKLE') or (str(DEFAULT_LOCAL_RAW_PICKLE) if DEFAULT_LOCAL_RAW_PICKLE.exists() else None)

SEED = 42
BATCH_SIZE = 32
NUM_WORKERS = 0
DEVICE = 'auto'
PRETRAINED = True
FREEZE_BACKBONE = True
NORMALIZE_IMAGENET = True
BACKBONE_INPUT_SIZE = 224
QUERY_CHUNK_SIZE = 512
MEMORY_CHUNK_SIZE = 4096
THRESHOLD_QUANTILE = 0.95
THRESHOLD_STRATEGY = 'validation_f1'
MAX_VALIDATION_FALSE_POSITIVE_RATE = None
SPLIT_CONFIG = DEFAULT_SPLIT_CONFIG.copy()
DATA_ROOT = Path(os.environ.get('WM811K_DATA_ROOT', str(PROJECT_ROOT / 'data'))).resolve()
OUTPUT_ROOT = Path(os.environ.get('WM811K_OUTPUT_ROOT', str(PROJECT_ROOT / 'artifacts'))).resolve()
ARTIFACT_ROOT = OUTPUT_ROOT / 'patchcore_wrn50_multilayer_120k_notebook4'
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

RUN_EXPERIMENTS = False
ACTIVE_GROUPS = ['coverage_sampling', 'image_size', 'layer_sweep']

DOWNSAMPLE_MAP = {'layer1': 4, 'layer2': 8, 'layer3': 16, 'layer4': 32}


def patches_per_image_for_layers(teacher_layers: list[str], backbone_input_size: int = BACKBONE_INPUT_SIZE) -> int:
    output_spatial = max(1, backbone_input_size // min(DOWNSAMPLE_MAP[layer] for layer in teacher_layers))
    return int(output_spatial * output_spatial)


def default_source_images(memory_bank_size: int, teacher_layers: list[str]) -> int:
    return int(math.ceil(memory_bank_size / patches_per_image_for_layers(teacher_layers)))


def make_experiment(
    *,
    name: str,
    group: str,
    image_size: int,
    teacher_layers: list[str],
    memory_bank_size: int = 50_000,
    topk_ratio: float = 0.05,
    reduction: str = 'topk_mean',
    memory_source_images: int | None = None,
    normal_only_memory_sampling: bool = True,
    note: str = '',
) -> dict[str, object]:
    teacher_layers = [str(layer).lower() for layer in teacher_layers]
    return {
        'name': name,
        'group': group,
        'image_size': int(image_size),
        'teacher_layers': teacher_layers,
        'memory_bank_size': int(memory_bank_size),
        'topk_ratio': float(topk_ratio),
        'reduction': str(reduction),
        'memory_source_images': memory_source_images,
        'normal_only_memory_sampling': bool(normal_only_memory_sampling),
        'note': str(note),
        'expected_patches_per_image': patches_per_image_for_layers(teacher_layers),
        'default_source_images': default_source_images(memory_bank_size, teacher_layers),
    }


EXPERIMENT_SPECS = [
    make_experiment(
        name='coverage__baseline_64_l23_auto_mixed',
        group='coverage_sampling',
        image_size=64,
        teacher_layers=['layer2', 'layer3'],
        memory_source_images=None,
        normal_only_memory_sampling=False,
        note='Current-style control: mixed source-image sampling with the default source-image count.',
    ),
    make_experiment(
        name='coverage__64_l23_normals_512src',
        group='coverage_sampling',
        image_size=64,
        teacher_layers=['layer2', 'layer3'],
        memory_source_images=512,
        normal_only_memory_sampling=True,
        note='Increase source-image coverage while forcing normal-only memory sampling.',
    ),
    make_experiment(
        name='coverage__64_l23_normals_2048src',
        group='coverage_sampling',
        image_size=64,
        teacher_layers=['layer2', 'layer3'],
        memory_source_images=2048,
        normal_only_memory_sampling=True,
        note='Main coverage candidate for the next pass.',
    ),
    make_experiment(
        name='coverage__64_l23_normals_8192src',
        group='coverage_sampling',
        image_size=64,
        teacher_layers=['layer2', 'layer3'],
        memory_source_images=8192,
        normal_only_memory_sampling=True,
        note='Aggressive coverage test to see whether more source wafers still help.',
    ),
    make_experiment(
        name='image__96_l23_normals_2048src',
        group='image_size',
        image_size=96,
        teacher_layers=['layer2', 'layer3'],
        memory_source_images=2048,
        normal_only_memory_sampling=True,
        note='Increase wafer-map detail while keeping the stronger 2048-source normal-only bank.',
    ),
    make_experiment(
        name='image__128_l23_normals_2048src',
        group='image_size',
        image_size=128,
        teacher_layers=['layer2', 'layer3'],
        memory_source_images=2048,
        normal_only_memory_sampling=True,
        note='Highest-detail image-size check before changing layers.',
    ),
    make_experiment(
        name='layer__96_l12_normals_2048src',
        group='layer_sweep',
        image_size=96,
        teacher_layers=['layer1', 'layer2'],
        memory_source_images=2048,
        normal_only_memory_sampling=True,
        note='Shallower feature mix to test compact defect sensitivity.',
    ),
    make_experiment(
        name='layer__96_l123_normals_2048src',
        group='layer_sweep',
        image_size=96,
        teacher_layers=['layer1', 'layer2', 'layer3'],
        memory_source_images=2048,
        normal_only_memory_sampling=True,
        note='Layer1 inclusion without dropping the current deeper features.',
    ),
    make_experiment(
        name='layer__128_l123_normals_2048src',
        group='layer_sweep',
        image_size=128,
        teacher_layers=['layer1', 'layer2', 'layer3'],
        memory_source_images=2048,
        normal_only_memory_sampling=True,
        note='Highest-detail plus layer1 inclusion stress test.',
    ),
]

plan_df = pd.DataFrame(EXPERIMENT_SPECS)
plan_df['teacher_layers_label'] = plan_df['teacher_layers'].apply(lambda layers: ' + '.join(layers))
plan_df['requested_source_images'] = plan_df['memory_source_images'].fillna(plan_df['default_source_images'])
plan_df.to_csv(ARTIFACT_ROOT / 'notebook4_experiment_plan.csv', index=False)


In [ ]:
display(
    plan_df[
        [
            'group',
            'name',
            'image_size',
            'teacher_layers_label',
            'memory_bank_size',
            'default_source_images',
            'requested_source_images',
            'normal_only_memory_sampling',
            'expected_patches_per_image',
            'note',
        ]
    ]
)

active_plan_df = plan_df[plan_df['group'].isin(ACTIVE_GROUPS)].reset_index(drop=True)
print(f'Notebook root: {NOTEBOOK_ROOT}')
print(f'Project root: {PROJECT_ROOT}')
print(f'Data root: {DATA_ROOT}')
print(f'Artifact root: {ARTIFACT_ROOT}')
print(f'Configured raw pickle: {RAW_PICKLE}')
print(f'RUN_EXPERIMENTS = {RUN_EXPERIMENTS}')
print(f'Active groups: {ACTIVE_GROUPS}')
print(f'Planned experiments in active groups: {len(active_plan_df)}')


In [ ]:
dataset_cache: dict[int, dict[str, object]] = {}


def load_split_resources(image_size: int, raw_pickle: Path) -> dict[str, object]:
    image_size = int(image_size)
    if image_size in dataset_cache:
        return dataset_cache[image_size]

    metadata_path = prepare_dataset(
        raw_pickle,
        DATA_ROOT,
        image_size,
        SPLIT_CONFIG,
        seed=SEED,
        overwrite=False,
    )
    metadata = pd.read_csv(metadata_path)
    resources = {
        'metadata_path': metadata_path,
        'metadata': metadata,
        'train_dataset': WaferArrayDataset(metadata_path, split='train', data_root=DATA_ROOT),
        'val_dataset': WaferArrayDataset(metadata_path, split='val', data_root=DATA_ROOT),
        'test_dataset': WaferArrayDataset(metadata_path, split='test', data_root=DATA_ROOT),
    }
    resources['val_loader'] = DataLoader(
        resources['val_dataset'],
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
    )
    resources['test_loader'] = DataLoader(
        resources['test_dataset'],
        batch_size=BATCH_SIZE,
        shuffle=False,
        num_workers=NUM_WORKERS,
    )
    dataset_cache[image_size] = resources
    return resources


def run_experiment(spec: dict[str, object], raw_pickle: Path, device: torch.device) -> tuple[dict[str, object], dict[str, object]]:
    resources = load_split_resources(int(spec['image_size']), raw_pickle)
    variant = {
        'name': str(spec['name']),
        'memory_bank_size': int(spec['memory_bank_size']),
        'reduction': str(spec['reduction']),
        'topk_ratio': float(spec['topk_ratio']),
        'memory_source_images': spec['memory_source_images'],
        'normal_only_memory_sampling': bool(spec['normal_only_memory_sampling']),
    }
    result = run_patchcore_variant(
        variant,
        train_dataset=resources['train_dataset'],
        val_loader=resources['val_loader'],
        test_loader=resources['test_loader'],
        batch_size=BATCH_SIZE,
        num_workers=NUM_WORKERS,
        device=device,
        output_dir=ARTIFACT_ROOT / str(spec['group']),
        seed=SEED,
        teacher_layers=list(spec['teacher_layers']),
        pretrained=PRETRAINED,
        freeze_backbone=FREEZE_BACKBONE,
        backbone_input_size=BACKBONE_INPUT_SIZE,
        normalize_imagenet=NORMALIZE_IMAGENET,
        threshold_quantile=THRESHOLD_QUANTILE,
        threshold_strategy=THRESHOLD_STRATEGY,
        max_validation_false_positive_rate=MAX_VALIDATION_FALSE_POSITIVE_RATE,
        query_chunk_size=QUERY_CHUNK_SIZE,
        memory_chunk_size=MEMORY_CHUNK_SIZE,
    )
    row = {
        **result['row'],
        'group': str(spec['group']),
        'image_size': int(spec['image_size']),
        'teacher_layers_label': ' + '.join(spec['teacher_layers']),
        'note': str(spec['note']),
    }
    return row, result


results_df = pd.DataFrame()
experiment_results: dict[str, dict[str, object]] = {}


In [ ]:
if not RUN_EXPERIMENTS:
    display(Markdown("""### Execution is disabled

Set `RUN_EXPERIMENTS = True` in the config cell when you want to launch the sweep."""))
else:
    set_seed(SEED)
    device = resolve_device(DEVICE)
    raw_pickle = Path(auto_find_raw_pickle(RAW_PICKLE)).resolve()
    print(f'Using device: {device}')
    print(f'Raw pickle: {raw_pickle}')

    active_specs = [spec for spec in EXPERIMENT_SPECS if spec['group'] in ACTIVE_GROUPS]
    rows = []

    for spec in active_specs:
        print(f"\n=== Running experiment: {spec['name']} ===")
        print(
            f"group={spec['group']} | image_size={spec['image_size']} | "
            f"layers={' + '.join(spec['teacher_layers'])} | "
            f"source_images={spec['memory_source_images']} | "
            f"normal_only={spec['normal_only_memory_sampling']}"
        )
        row, result = run_experiment(spec, raw_pickle=raw_pickle, device=device)
        experiment_results[str(spec['name'])] = result
        rows.append(row)

    results_df = pd.DataFrame(rows).sort_values(['f1', 'auroc', 'auprc'], ascending=False).reset_index(drop=True)
    results_df.to_csv(ARTIFACT_ROOT / 'notebook4_results.csv', index=False)

    bundle_summary = {
        'threshold_strategy': THRESHOLD_STRATEGY,
        'max_validation_false_positive_rate': MAX_VALIDATION_FALSE_POSITIVE_RATE,
        'batch_size': BATCH_SIZE,
        'device': str(device),
        'raw_pickle': str(raw_pickle),
        'active_groups': ACTIVE_GROUPS,
        'experiments': results_df.to_dict(orient='records'),
    }
    (ARTIFACT_ROOT / 'notebook4_summary.json').write_text(json.dumps(bundle_summary, indent=2), encoding='utf-8')

    display(results_df)


In [ ]:
if results_df.empty:
    display(Markdown("""### No experiment results yet

The plan table above is ready. Once the sweep runs, this section will summarize the best result in each group."""))
else:
    best_by_group_df = (
        results_df.sort_values(['f1', 'auroc', 'auprc'], ascending=False)
        .groupby('group', as_index=False)
        .first()
    )
    display(best_by_group_df[[
        'group',
        'name',
        'image_size',
        'teacher_layers_label',
        'memory_source_images_requested',
        'normal_only_memory_sampling',
        'precision',
        'recall',
        'f1',
        'auroc',
        'auprc',
        'false_positive_rate',
    ]].round(4))

    plot_df = results_df.copy()
    plot_df['label'] = (
        plot_df['name']
        + '\nimg=' + plot_df['image_size'].astype(str)
        + ', layers=' + plot_df['teacher_layers_label']
    )

    fig, axes = plt.subplots(1, 3, figsize=(18, 7))
    axes[0].barh(plot_df['label'], plot_df['f1'], color='#277da1')
    axes[0].set_title('Notebook 4 Sweep: F1')
    axes[0].set_xlabel('F1')
    axes[0].invert_yaxis()

    axes[1].barh(plot_df['label'], plot_df['auroc'], color='#90be6d')
    axes[1].set_title('Notebook 4 Sweep: AUROC')
    axes[1].set_xlabel('AUROC')
    axes[1].invert_yaxis()

    axes[2].barh(plot_df['label'], plot_df['auprc'], color='#f9844a')
    axes[2].set_title('Notebook 4 Sweep: AUPRC')
    axes[2].set_xlabel('AUPRC')
    axes[2].invert_yaxis()

    plt.tight_layout()
    plt.show()


In [ ]:
if results_df.empty:
    guidance = """
### Recommended Execution Order

1. Run `coverage_sampling` first. That isolates whether larger source-image coverage and normal-only sampling help before we spend compute on bigger images or shallower layers.
2. If `coverage__64_l23_normals_2048src` beats the current-style control, run `image_size` next to compare `96` and `128` against that stronger memory-bank setup.
3. Run `layer_sweep` last. Once we know whether `96` or `128` is the better input size, we can judge whether `layer1` adds signal or just more noise and compute.
"""
    display(Markdown(guidance))
else:
    best_row = results_df.sort_values(['f1', 'auroc', 'auprc'], ascending=False).iloc[0]
    message = f"""
### Working Recommendation

- Best overall experiment so far: **{best_row['name']}**
- Image size: `{int(best_row['image_size'])}`
- Teacher layers: `{best_row['teacher_layers_label']}`
- Requested source images: `{best_row['memory_source_images_requested']}`
- Normal-only memory sampling: `{bool(best_row['normal_only_memory_sampling'])}`
- Test precision / recall / F1: `{best_row['precision']:.4f} / {best_row['recall']:.4f} / {best_row['f1']:.4f}`
- AUROC / AUPRC: `{best_row['auroc']:.4f} / {best_row['auprc']:.4f}`

If the winning config comes from the coverage sweep, the next move should be to keep that stronger memory-bank recipe and continue with the image-size and `layer1` comparisons. If the winner already includes `96 / 128` or `layer1`, that should become the new control for the next remote training run.
"""
    display(Markdown(message))
